In [1]:
import pandas as pd

In [2]:
# Import the CSV, convert Timestamp to datetime, and convert Converted to a boolean column (True/False instead of Yes/No).
df = pd.read_csv('web_traffic.csv')
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['Converted'] = df['Converted'].apply(lambda x: False if x == 'No' else True)
df

,Session_ID,User_ID,Page,Timestamp,Time_On_Page_Sec,Device,Converted
0,S001,U101,Home,2024-03-01 08:15:00,45,Mobile,False
1,S001,U101,Product,2024-03-01 08:16:00,120,Mobile,False
2,S001,U101,Cart,2024-03-01 08:18:30,60,Mobile,True
3,S002,U102,Home,2024-03-01 09:00:00,30,Desktop,False
4,S002,U102,Product,2024-03-01 09:01:00,90,Desktop,False
5,S002,U102,Home,2024-03-01 09:03:00,20,Desktop,False
6,S003,U103,Product,2024-03-01 10:30:00,200,Tablet,False
7,S003,U103,Cart,2024-03-01 10:34:00,45,Tablet,False
8,S003,U103,Checkout,2024-03-01 10:35:00,30,Tablet,True
9,S004,U101,Home,2024-03-02 11:00:00,50,Desktop,False


In [3]:
# Calculate the total Time_On_Page_Sec per session (Session_ID). Then find the average session duration by Device type. Display the result sorted descending.
sessiontime = df.groupby(['Session_ID', 'Device'])['Time_On_Page_Sec'].sum().reset_index(name='Session_Duration')
devicetime = sessiontime.groupby('Device')['Session_Duration'].mean().sort_values(ascending=False).reset_index(name = 'Device_Session')

devicetime

,Device,Device_Session
0,Tablet,275.000000
1,Desktop,256.666667
2,Mobile,151.666667


In [4]:
# Calculate the conversion rate by Device type. A session is considered "converted" if it contains at least one Converted == True row. The conversion rate is the number of converted sessions divided by total sessions for that device. Display as a percentage rounded to 1 decimal place.
conversion = df.groupby(['Session_ID', 'Device'])['Converted'].any().reset_index(name = 'Did_They_Convert')
rate = conversion.groupby('Device')['Did_They_Convert'].mean().reset_index(name = 'Conversion_Rate')
rate['Conversion_Rate'] = (rate['Conversion_Rate'] * 100).round(1)

rate

,Device,Conversion_Rate
0,Desktop,33.3
1,Mobile,66.7
2,Tablet,100.0


In [5]:
# Find each user's most visited page (the page they viewed the most times across all sessions). If there's a tie, any one of the tied pages is fine. Display the result.
page_counts = df.groupby(['User_ID', 'Page']).size().reset_index(name='Visit_Count')
top_pages = page_counts.sort_values('Visit_Count', ascending=False).drop_duplicates(subset='User_ID')

top_pages

,User_ID,Page,Visit_Count
0,U101,Cart,2
5,U102,Home,2
10,U104,Home,2
13,U105,Product,2
7,U103,Cart,1


In [6]:
# Create a column called Pages_In_Session that shows the total number of page views in that row's session. Then filter to show only sessions where the user visited more than 3 pages.
df['Pages_In_Session'] = df.groupby('Session_ID')['Page'].transform('count')
df[df['Pages_In_Session'] > 3]

,Session_ID,User_ID,Page,Timestamp,Time_On_Page_Sec,Device,Converted,Pages_In_Session
9,S004,U101,Home,2024-03-02 11:00:00,50,Desktop,False,4
10,S004,U101,Product,2024-03-02 11:01:00,180,Desktop,False,4
11,S004,U101,Cart,2024-03-02 11:04:30,90,Desktop,False,4
12,S004,U101,Checkout,2024-03-02 11:06:30,40,Desktop,True,4
17,S007,U105,Home,2024-03-03 09:00:00,25,Desktop,False,4
18,S007,U105,Product,2024-03-03 09:01:00,80,Desktop,False,4
19,S007,U105,Product,2024-03-03 09:02:30,95,Desktop,False,4
20,S007,U105,Cart,2024-03-03 09:04:30,70,Desktop,False,4
